In [1]:
import serial.tools.list_ports
import time
import serial



def find_arduino_port():
    ports = serial.tools.list_ports.comports()
    for port in ports:
        if 'usbmodem' in port.device or 'usbserial' in port.device:
            return port.device
    raise IOError("Arduino not found. Please check the connection.")

# Use this function to set the port
arduino_port = find_arduino_port()
print(f"Connected to Arduino on {arduino_port}")

# Then connect as usual
arduino = serial.Serial(arduino_port, 9600, timeout=1)
time.sleep(2)  # Give the Arduino time to reset


Connected to Arduino on /dev/cu.usbmodem11401


In [ ]:
import pygame
import time
import random

# Initialize Pygame for controller input
pygame.init()
pygame.joystick.init()

if pygame.joystick.get_count() == 0:
    print("No controller detected!")
    exit()

joystick = pygame.joystick.Joystick(0)
joystick.init()
print(f"Connected to {joystick.get_name()}")

# Constants
DEAD_ZONE    = 0.1       # Ignore tiny joystick movements
MANUAL_SPEED = 5         # Incremental steps per update (manual mode)
AUTO_SPEED   = 5         # Constant steps per update (auto mode)
BOX_HALF_SIZE = 10       # Auto mode box: ±50 units around the center
X_MIN, X_MAX = -400, 400  # Absolute boundaries for X
Y_MIN, Y_MAX = -400, 400  # Absolute boundaries for Y

AUTO_MODE = False  # Global auto mode flag

# Global position (manual and auto share this as the current laser position)
prev_x = 0
prev_y = 0


auto_center_x = 0
auto_center_y = 0

# Auto mode variables
auto_x_min = None
auto_x_max = None
auto_y_min = None
auto_y_max = None
target_x = 0
target_y = 0

def clamp(val, min_val, max_val):
    return max(min_val, min(max_val, val))

def move_laser(new_x, new_y):
    """Clamp the new position to allowed boundaries and, if changed, print movement commands."""
    global prev_x, prev_y
    new_x = clamp(new_x, X_MIN, X_MAX)
    new_y = clamp(new_y, Y_MIN, Y_MAX)
    if new_x != prev_x:
        print(f"MOVE_X {new_x}")
        arduino.write(f"MOVE_X {new_x}\n".encode())

    if new_y != prev_y:
        print(f"MOVE_Y {new_y}")
        arduino.write(f"MOVE_Y {new_y}\n".encode())

    prev_x, prev_y = new_x, new_y

def manual_control():
    """Increment the current position based on joystick input (relative movement)."""
    global prev_x, prev_y
    x = joystick.get_axis(0)  # Left stick X-axis (-1 to 1)
    y = joystick.get_axis(1)  # Left stick Y-axis (-1 to 1)
    
    # Use incremental movement if outside the dead zone
    if abs(x) > DEAD_ZONE:
        new_x = prev_x + int(x * MANUAL_SPEED)
    else:
        new_x = prev_x
    if abs(y) > DEAD_ZONE:
        new_y = prev_y + int(y * MANUAL_SPEED)
    else:
        new_y = prev_y

    move_laser(new_x, new_y)
    
    # Handle laser toggle on button 1 (B button)
    if joystick.get_button(1):
        print("LASER TOGGLE")
        arduino.write(b"LASER_TOGGLE\n")
        time.sleep(0.2)

def auto_mode_movement():
    """Move the laser toward a target at constant speed.
       When close, snap to it and pick a new random target within the auto box."""
    global prev_x, prev_y, target_x, target_y, auto_x_min, auto_x_max, auto_y_min, auto_y_max
    dx = target_x - prev_x
    dy = target_y - prev_y
    distance = (dx**2 + dy**2) ** 0.5

    if distance < AUTO_SPEED:
        # Snap to the target and choose a new one
        move_laser(target_x, target_y)
        target_x = random.randint(auto_x_min, auto_x_max)
        target_y = random.randint(auto_y_min, auto_y_max)
        print(f"New target: X {target_x}, Y {target_y}")
    else:
        # Compute constant speed movement toward the target
        move_x = AUTO_SPEED * dx / distance
        move_y = AUTO_SPEED * dy / distance
        new_x = int(prev_x + move_x)
        new_y = int(prev_y + move_y)
        move_laser(new_x, new_y)
    time.sleep(0.05)

def toggle_auto_mode():
    """Toggle auto mode. When turning it on, record a ±50 box centered on the current position."""
    global AUTO_MODE, auto_x_min, auto_x_max, auto_y_min, auto_y_max, target_x, target_y, prev_x, prev_y
    AUTO_MODE = not AUTO_MODE
    if AUTO_MODE:
        # Set the auto mode box boundaries using the current position as the center
        global auto_center_x, auto_center_y
        auto_center_x = prev_x
        auto_center_y = prev_y
        auto_x_min = auto_center_x - BOX_HALF_SIZE
        auto_x_max = auto_center_x + BOX_HALF_SIZE
        auto_y_min = auto_center_y - BOX_HALF_SIZE
        auto_y_max = auto_center_y + BOX_HALF_SIZE

        print(f"Auto mode ON. Box: X [{auto_x_min}, {auto_x_max}], Y [{auto_y_min}, {auto_y_max}]")
        # Pick an initial target within that box
        target_x = random.randint(auto_x_min, auto_x_max)
        target_y = random.randint(auto_y_min, auto_y_max)
        print(f"Initial target: X {target_x}, Y {target_y}")
    else:
        print("Auto mode OFF")

while True:
    pygame.event.pump()

    hat = joystick.get_hat(0)
    if hat[1] == 1:
        BOX_HALF_SIZE += 5
        print(f"Box size increased to ±{BOX_HALF_SIZE}")
        if AUTO_MODE:
            auto_x_min = auto_center_x - BOX_HALF_SIZE
            auto_x_max = auto_center_x + BOX_HALF_SIZE
            auto_y_min = auto_center_y - BOX_HALF_SIZE
            auto_y_max = auto_center_y + BOX_HALF_SIZE
            print(f"Auto box updated: X [{auto_x_min}, {auto_x_max}], Y [{auto_y_min}, {auto_y_max}]")
        time.sleep(0.2)
    elif hat[1] == -1:
        BOX_HALF_SIZE = max(5, BOX_HALF_SIZE - 5)
        print(f"Box size decreased to ±{BOX_HALF_SIZE}")
        if AUTO_MODE:
            auto_x_min = auto_center_x - BOX_HALF_SIZE
            auto_x_max = auto_center_x + BOX_HALF_SIZE
            auto_y_min = auto_center_y - BOX_HALF_SIZE
            auto_y_max = auto_center_y + BOX_HALF_SIZE
            print(f"Auto box updated: X [{auto_x_min}, {auto_x_max}], Y [{auto_y_min}, {auto_y_max}]")
        time.sleep(0.2)

    
    # Check for Y button (assumed to be button 3) to toggle auto mode
    if joystick.get_button(3):
        toggle_auto_mode()
        time.sleep(0.2)  # Delay to prevent rapid toggling

    # If auto mode is active, run auto mode movement; otherwise, use manual control.
    if AUTO_MODE:
        auto_mode_movement()
    else:
        manual_control()
    
    pygame.time.delay(50)


pygame 2.6.1 (SDL 2.28.4, Python 3.12.2)
Hello from the pygame community. https://www.pygame.org/contribute.html
Connected to 8BitDo Ultimate 2C Wireless
MOVE_X -2
MOVE_X -7
MOVE_X -12
MOVE_X -17
MOVE_X -22
MOVE_X -27
MOVE_X -32
MOVE_X -37
MOVE_X -42
MOVE_X -44
MOVE_X -40
MOVE_X -36
MOVE_X -32
MOVE_X -28
MOVE_X -24
MOVE_X -20
MOVE_X -16
MOVE_Y -1
LASER TOGGLE
MOVE_X -21
MOVE_X -26
MOVE_X -31
MOVE_X -36
MOVE_X -41
MOVE_X -46
MOVE_X -51
MOVE_X -56
MOVE_X -61
MOVE_X -66
MOVE_X -71
MOVE_Y -2
MOVE_X -76
MOVE_Y -3
MOVE_X -81
MOVE_Y -4
MOVE_X -86
MOVE_Y -5
MOVE_X -91
MOVE_Y -6
MOVE_X -96
MOVE_Y -8
MOVE_X -100
MOVE_Y -11
MOVE_X -103
MOVE_Y -16
MOVE_X -106
MOVE_Y -21
MOVE_X -109
MOVE_Y -26
MOVE_X -111
MOVE_Y -31
MOVE_X -113
MOVE_Y -36
MOVE_X -115
MOVE_Y -41
MOVE_X -117
MOVE_Y -46
MOVE_X -119
MOVE_Y -51
MOVE_X -121
MOVE_Y -56
MOVE_X -123
MOVE_Y -61
MOVE_X -125
MOVE_Y -66
MOVE_X -128
MOVE_Y -71
MOVE_X -129
MOVE_X -134
MOVE_Y -72
MOVE_X -139
MOVE_Y -73
MOVE_X -144
MOVE_Y -74
MOVE_X -149
MOVE_X -15

In [2]:
##UNCOMMENT IF USING SLIDERS

# import matplotlib.pyplot as plt
# import ipywidgets as widgets
# from IPython.display import display
# import serial
# import time
# import matplotlib
# print(matplotlib.get_backend())

# # Install necessary libraries (if not already installed)
# # Uncomment if needed:
# # !pip install matplotlib ipywidgets pyserial

# # ----------------------------
# # Motor Control Functions
# # ----------------------------

# # Define min/max limits for safety
# X_MIN, X_MAX = -400, 400
# Y_MIN, Y_MAX = -400, 400

# def move_x(steps):
#     """Move X motor to specified position within limits."""
#     if X_MIN <= steps <= X_MAX:
#         arduino.write(f"MOVE_X {steps}\n".encode())
#         print(f"Sent to X: {steps}")
#     else:
#         print(f"X position {steps} out of range!")


# def move_y(steps):
#     """Move Y motor to specified position within limits (inverted direction)."""
#     inverted_steps = -steps  # Invert Y-axis direction
#     if Y_MIN <= steps <= Y_MAX:
#         arduino.write(f"MOVE_Y {inverted_steps}\n".encode())
#         print(f"Sent to Y: {steps} (inverted to {inverted_steps})")
#     else:
#         print(f"Y position {steps} out of range!")


# def reset_motors():
#     """Reset both motors to zero."""
#     arduino.write(b"RESET\n")
#     print("Motors reset to 0")
    
#     # Reset sliders to 0
#     x_slider.value = 0
#     y_slider.value = 0


In [20]:
# Create sliders for X and Y positions
x_slider = widgets.IntSlider(value=0, min=X_MIN, max=X_MAX, step=10, description='X Position:')
y_slider = widgets.IntSlider(value=0, min=Y_MIN, max=Y_MAX, step=10, description='Y Position:')

# Output widget to display logs
output = widgets.Output()

# Define what happens when sliders are moved
def on_x_slider_change(change):
    if change['name'] == 'value':
        move_x(change['new'])


def on_y_slider_change(change):
    if change['name'] == 'value':
        move_y(change['new'])

# Attach event listeners to sliders
x_slider.observe(on_x_slider_change, names='value')
y_slider.observe(on_y_slider_change, names='value')

# Reset button
reset_button = widgets.Button(description="Reset Motors")

def reset_motors_callback(b):
    with output:
        output.clear_output()
        reset_motors()

reset_button.on_click(reset_motors_callback)

# Display the UI
display(widgets.VBox([x_slider, y_slider, reset_button, output]))